# Tempreture Recomendation Transformer Training

This notebook trains a PyTorch Transformer regressor for HVAC setpoint recommendation. It mounts Google Drive, reads `temperatureData.csv`, builds recent room-level time-series sequences, learns an efficient recommended setpoint target, and saves model artifacts back to Drive.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive')
CANDIDATE_DATA_PATHS = [
    DRIVE_ROOT / 'VisualizationApp' / 'temperatureData.csv',
    DRIVE_ROOT / 'Data' / 'temperatureData.csv',
    DRIVE_ROOT / 'temperatureData.csv',
]

DATA_CSV = next((p for p in CANDIDATE_DATA_PATHS if p.exists()), CANDIDATE_DATA_PATHS[0])
OUT_DIR = DRIVE_ROOT / 'VisualizationApp' / 'AIModelsAndAlgorithms' / 'TempretureRecomendation' / 'transformer'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_CSV:', DATA_CSV)
print('DATA_CSV exists:', DATA_CSV.exists())
print('OUT_DIR:', OUT_DIR)

if not DATA_CSV.exists():
    raise FileNotFoundError(
        'Could not find temperatureData.csv in the default Drive locations. '
        'Set DATA_CSV manually to your file path, then rerun this cell.'
    )


DATA_CSV: /content/drive/MyDrive/VisualizationApp/temperatureData.csv
DATA_CSV exists: True
OUT_DIR: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer


In [3]:
import json
import math
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)


device: cuda


In [4]:
# Training controls. Start smaller for a smoke test, then increase for full training.
MAX_ROWS = None
MAX_SEQUENCES = 500_000
SEQ_LEN = 24             # 24 samples = 2 hours if data is every 5 minutes.
STRIDE = 3
BATCH_SIZE = 512
EPOCHS = 20
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
D_MODEL = 96
N_HEADS = 4
N_LAYERS = 2
DIM_FEEDFORWARD = 192
DROPOUT = 0.15
PATIENCE = 4
SETPOINT_MIN = 16.0
SETPOINT_MAX = 28.0
PREDICTION_HORIZON_MINUTES = 5


In [5]:
USECOLS = [
    'timestamp', 'room_number', 'floor', 'facade', 'room_type', 'size_m2',
    'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'hvac_mode',
    'ac_persona', 'occupant_state', 'pir_persona', 'room_state', 'pir_motion', 'guest_id',
]

BASE_NUMERIC_COLS = [
    'floor_scaled', 'size_scaled', 'outside_temp_scaled', 'room_temp_scaled',
    'setpoint_scaled', 'ideal_temp_scaled', 'temp_error_scaled', 'comfort_error_scaled',
    'pir_motion', 'has_guest', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
]
CAT_COLS = [
    'facade', 'room_type', 'hvac_mode', 'occupant_state', 'pir_persona',
    'occupancy_prediction', 'temperature_persona_prediction',
]

PERSONA_POLICY = {
    'AlwaysOnComfort': {'cooling': 22.5, 'heating': 22.0, 'idle': 22.5},
    'EnergySaver': {'cooling': 26.0, 'heating': 19.0, 'idle': 24.0},
    'Reactive': {'cooling': 23.5, 'heating': 21.5, 'idle': 23.0},
    'Preconditioning': {'cooling': 23.0, 'heating': 21.0, 'idle': 23.0},
    'Housekeeping': {'cooling': 24.0, 'heating': 20.0, 'idle': 23.0},
    'Unknown': {'cooling': 24.0, 'heating': 20.0, 'idle': 23.0},
}

def cyclic(values, period):
    radians = 2 * np.pi * values / period
    return np.sin(radians), np.cos(radians)

def scale_temp(series):
    return ((pd.to_numeric(series, errors='coerce').fillna(0).clip(-20, 50) + 20) / 70).astype('float32')

def infer_target_mode(row):
    mode = str(row.get('hvac_mode') or '').lower()
    room_temp = float(row.get('room_temp') or 0.0)
    ideal_temp = float(row.get('ideal_temp') or 22.0)
    outside_temp = float(row.get('outside_temp') or ideal_temp)
    if mode in {'cooling', 'heating'}:
        return mode
    if room_temp >= ideal_temp + 0.7 or outside_temp >= 27:
        return 'cooling'
    if room_temp <= ideal_temp - 0.7 or outside_temp <= 12:
        return 'heating'
    return 'idle'

def recommended_setpoint(row):
    occupancy = str(row.get('occupancy_prediction') or 'Unknown')
    persona = str(row.get('temperature_persona_prediction') or 'Unknown')
    persona = persona if persona in PERSONA_POLICY else 'Unknown'
    target_mode = infer_target_mode(row)
    current_setpoint = float(row.get('setpoint') or 22.0)
    ideal_temp = float(row.get('ideal_temp') or 22.0)

    if occupancy == 'Vacant':
        target = 28.0 if target_mode == 'cooling' else 17.0 if target_mode == 'heating' else 25.0
    elif occupancy == 'Cleaning' or persona == 'Housekeeping':
        target = PERSONA_POLICY['Housekeeping'][target_mode]
    else:
        target = PERSONA_POLICY[persona][target_mode]

    # Keep the recommendation realistic: do not jump too far from the current setpoint in one 5-minute step.
    target = 0.65 * target + 0.25 * ideal_temp + 0.10 * current_setpoint
    target = np.clip(round(target * 2) / 2, SETPOINT_MIN, SETPOINT_MAX)
    return float(target)


In [6]:
def load_and_prepare(data_csv, max_rows=None):
    df = pd.read_csv(data_csv, usecols=USECOLS, parse_dates=['timestamp'], nrows=max_rows)
    df = df.dropna(subset=['timestamp']).copy()
    df = df.sort_values(['room_number', 'timestamp']).reset_index(drop=True)

    for col in ['floor', 'size_m2', 'outside_temp', 'room_temp', 'setpoint', 'ideal_temp', 'pir_motion']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['guest_id'] = pd.to_numeric(df['guest_id'], errors='coerce')

    df['occupancy_prediction'] = df['room_state'].fillna('Unknown').astype(str)
    df['temperature_persona_prediction'] = df['ac_persona'].fillna('Unknown').astype(str)
    for col in ['facade', 'room_type', 'hvac_mode', 'occupant_state', 'pir_persona', 'occupancy_prediction', 'temperature_persona_prediction']:
        df[col] = df[col].fillna('Unknown').astype(str)

    df['floor_scaled'] = df['floor'].clip(0, 30) / 30.0
    df['size_scaled'] = df['size_m2'].clip(0, 120) / 120.0
    df['outside_temp_scaled'] = scale_temp(df['outside_temp'])
    df['room_temp_scaled'] = scale_temp(df['room_temp'])
    df['setpoint_scaled'] = scale_temp(df['setpoint'])
    df['ideal_temp_scaled'] = scale_temp(df['ideal_temp'])
    df['temp_error_scaled'] = ((df['room_temp'] - df['setpoint']).clip(-20, 20) + 20) / 40.0
    df['comfort_error_scaled'] = ((df['room_temp'] - df['ideal_temp']).clip(-20, 20) + 20) / 40.0
    df['pir_motion'] = df['pir_motion'].clip(0, 1).astype('float32')
    df['has_guest'] = df['guest_id'].notna().astype('float32')
    df['hour_sin'], df['hour_cos'] = cyclic(df['timestamp'].dt.hour, 24)
    df['dow_sin'], df['dow_cos'] = cyclic(df['timestamp'].dt.dayofweek, 7)

    for col in BASE_NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('float32')

    df['recommended_setpoint'] = df.apply(recommended_setpoint, axis=1).astype('float32')
    df['target_scaled'] = ((df['recommended_setpoint'] - SETPOINT_MIN) / (SETPOINT_MAX - SETPOINT_MIN)).astype('float32')
    return df.reset_index(drop=True)

df = load_and_prepare(DATA_CSV, MAX_ROWS)
print('rows:', len(df))
print('time range:', df['timestamp'].min(), 'to', df['timestamp'].max())
print(df[['room_temp', 'setpoint', 'ideal_temp', 'hvac_mode', 'occupancy_prediction', 'temperature_persona_prediction', 'recommended_setpoint']].head())


rows: 5212900
time range: 2022-01-06 00:00:00 to 2022-07-06 00:00:00
   room_temp  setpoint  ideal_temp hvac_mode occupancy_prediction  \
0      15.91      15.0        22.0       off               Vacant   
1      15.82      15.0        22.0       off               Vacant   
2      15.73      15.0        22.0       off               Vacant   
3      15.65      15.0        22.0       off               Vacant   
4      15.56      15.0        22.0       off               Vacant   

  temperature_persona_prediction  recommended_setpoint  
0                        Unknown                  18.0  
1                        Unknown                  18.0  
2                        Unknown                  18.0  
3                        Unknown                  18.0  
4                        Unknown                  18.0  


In [7]:
category_values = {}
for col in CAT_COLS:
    values = sorted(df[col].fillna('Unknown').astype(str).unique().tolist())
    if 'Unknown' not in values:
        values.append('Unknown')
    category_values[col] = values

feature_names = list(BASE_NUMERIC_COLS)
for col, values in category_values.items():
    feature_names.extend([f'{col}={value}' for value in values])

def encode_features(frame):
    parts = [frame[BASE_NUMERIC_COLS].to_numpy(dtype='float32')]
    for col, values in category_values.items():
        mapping = {value: i for i, value in enumerate(values)}
        unknown_idx = mapping.get('Unknown', 0)
        ids = frame[col].fillna('Unknown').astype(str).map(mapping).fillna(unknown_idx).astype(int).to_numpy()
        onehot = np.zeros((len(frame), len(values)), dtype='float32')
        onehot[np.arange(len(frame)), ids] = 1.0
        parts.append(onehot)
    return np.concatenate(parts, axis=1)

encoded = encode_features(df)
targets = df['target_scaled'].to_numpy(dtype='float32')
print('feature count:', encoded.shape[1])
print('categories:', {k: len(v) for k, v in category_values.items()})


feature count: 49
categories: {'facade': 5, 'room_type': 4, 'hvac_mode': 4, 'occupant_state': 7, 'pir_persona': 5, 'occupancy_prediction': 4, 'temperature_persona_prediction': 6}


In [8]:
def build_sequences(frame, encoded_features, targets, seq_len, stride=1, max_sequences=None):
    frame = frame.reset_index(drop=True)
    sequences = []
    y = []
    end_indices = []

    for _, group in frame.groupby('room_number', sort=False):
        idx = group.index.to_numpy(dtype=np.int64)
        if len(idx) < seq_len:
            continue
        for end in range(seq_len - 1, len(idx), stride):
            window_idx = idx[end - seq_len + 1:end + 1]
            sequences.append(encoded_features[window_idx])
            y.append(targets[idx[end]])
            end_indices.append(idx[end])
            if max_sequences and len(sequences) >= max_sequences:
                return np.stack(sequences).astype('float32'), np.array(y, dtype='float32'), np.array(end_indices, dtype=np.int64)

    if not sequences:
        raise ValueError('No sequences were built. Reduce SEQ_LEN or check your data.')
    return np.stack(sequences).astype('float32'), np.array(y, dtype='float32'), np.array(end_indices, dtype=np.int64)

x, y, end_indices = build_sequences(df, encoded, targets, SEQ_LEN, STRIDE, MAX_SEQUENCES)
print('x:', x.shape, 'y:', y.shape)


x: (500000, 24, 49) y: (500000,)


In [9]:
sequence_times = df.loc[end_indices, 'timestamp'].reset_index(drop=True)
order = np.argsort(sequence_times.to_numpy())
x = x[order]
y = y[order]
end_indices = end_indices[order]

n = len(x)
train_end = int(n * 0.80)
val_end = int(n * 0.90)
x_train, y_train = x[:train_end], y[:train_end]
x_val, y_val = x[train_end:val_end], y[train_end:val_end]
x_test, y_test = x[val_end:], y[val_end:]
test_indices = end_indices[val_end:]

print('train:', x_train.shape, 'val:', x_val.shape, 'test:', x_test.shape)
print('split times:', sequence_times.iloc[train_end], sequence_times.iloc[val_end])

train_loader = DataLoader(TensorDataset(torch.from_numpy(x_train), torch.from_numpy(y_train)), batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(TensorDataset(torch.from_numpy(x_val), torch.from_numpy(y_val)), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(TensorDataset(torch.from_numpy(x_test), torch.from_numpy(y_test)), batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())


train: (400000, 24, 49) val: (50000, 24, 49) test: (50000, 24, 49)
split times: 2022-01-11 10:10:00 2022-06-19 09:40:00


In [10]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TempretureRecomendationTransformer(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, n_layers, dim_feedforward, dropout):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model // 2, 1))

    def forward(self, x):
        hidden = self.input_projection(x)
        hidden = self.position(hidden)
        hidden = self.encoder(hidden)
        last_token = hidden[:, -1, :]
        return torch.sigmoid(self.head(last_token)).squeeze(-1)

model = TempretureRecomendationTransformer(x_train.shape[-1], D_MODEL, N_HEADS, N_LAYERS, DIM_FEEDFORWARD, DROPOUT).to(DEVICE)
criterion = nn.SmoothL1Loss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
print(model)


/tmp/ipykernel_7726/1343508189.py:20: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)


TempretureRecomendationTransformer(
  (input_projection): Linear(in_features=49, out_features=96, bias=True)
  (position): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=96, out_features=96, bias=True)
        )
        (linear1): Linear(in_features=96, out_features=192, bias=True)
        (dropout): Dropout(p=0.15, inplace=False)
        (linear2): Linear(in_features=192, out_features=96, bias=True)
        (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.15, inplace=False)
        (dropout2): Dropout(p=0.15, inplace=False)
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=96, out_features=48, bias=True)
    (2): G

In [11]:
def run_epoch(loader, train=False):
    model.train(train)
    total_loss = 0.0
    total_count = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            pred = model(xb)
            loss = criterion(pred, yb)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total_loss += loss.item() * len(xb)
        total_count += len(xb)
    return total_loss / max(total_count, 1)

best_val = float('inf')
best_state = None
epochs_without_improvement = 0
history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
    print(f'epoch {epoch:02d} train_loss={train_loss:.5f} val_loss={val_loss:.5f}')
    if val_loss < best_val - 1e-5:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print('early stopping')
            break
if best_state is not None:
    model.load_state_dict(best_state)


epoch 01 train_loss=0.00324 val_loss=0.00147
epoch 02 train_loss=0.00128 val_loss=0.00111
epoch 03 train_loss=0.00113 val_loss=0.00112
epoch 04 train_loss=0.00103 val_loss=0.00233
epoch 05 train_loss=0.00096 val_loss=0.00104
epoch 06 train_loss=0.00089 val_loss=0.00089
epoch 07 train_loss=0.00084 val_loss=0.00099
epoch 08 train_loss=0.00084 val_loss=0.00092
epoch 09 train_loss=0.00078 val_loss=0.00085
epoch 10 train_loss=0.00074 val_loss=0.00080
epoch 11 train_loss=0.00074 val_loss=0.00069
epoch 12 train_loss=0.00072 val_loss=0.00067
epoch 13 train_loss=0.00069 val_loss=0.00074
epoch 14 train_loss=0.00068 val_loss=0.00066
epoch 15 train_loss=0.00064 val_loss=0.00073
epoch 16 train_loss=0.00065 val_loss=0.00060
epoch 17 train_loss=0.00062 val_loss=0.00088
epoch 18 train_loss=0.00062 val_loss=0.00079
epoch 19 train_loss=0.00059 val_loss=0.00056
epoch 20 train_loss=0.00062 val_loss=0.00114


In [12]:
@torch.no_grad()
def predict(loader):
    model.eval()
    preds = []
    actuals = []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        preds.append(model(xb).detach().cpu().numpy())
        actuals.append(yb.numpy())
    return np.concatenate(preds), np.concatenate(actuals)

pred_scaled, actual_scaled = predict(test_loader)
pred_setpoints = np.clip(pred_scaled * (SETPOINT_MAX - SETPOINT_MIN) + SETPOINT_MIN, SETPOINT_MIN, SETPOINT_MAX)
actual_setpoints = actual_scaled * (SETPOINT_MAX - SETPOINT_MIN) + SETPOINT_MIN
mae = mean_absolute_error(actual_setpoints, pred_setpoints)
rmse = float(np.sqrt(mean_squared_error(actual_setpoints, pred_setpoints)))
r2 = r2_score(actual_setpoints, pred_setpoints)
print('MAE:', mae)
print('RMSE:', rmse)
print('R2:', r2)


MAE: 0.21713000535964966
RMSE: 0.30966111785807504
R2: 0.9619849920272827


In [13]:
model_path = OUT_DIR / 'tempreture_recomendation_transformer.pt'
metadata_path = OUT_DIR / 'tempreture_recomendation_transformer_metadata.json'
report_path = OUT_DIR / 'tempreture_recomendation_transformer_report.txt'
predictions_path = OUT_DIR / 'tempreture_recomendation_transformer_sample_predictions.csv'
zip_path = OUT_DIR / 'tempreture_recomendation_transformer_outputs.zip'

metadata = {
    'model_type': 'transformer_regressor',
    'task': 'tempreture_recomendation',
    'prediction_horizon_minutes': PREDICTION_HORIZON_MINUTES,
    'target': 'recommended_setpoint',
    'setpoint_min': SETPOINT_MIN,
    'setpoint_max': SETPOINT_MAX,
    'sequence_length': SEQ_LEN,
    'stride': STRIDE,
    'feature_names': feature_names,
    'base_numeric_cols': BASE_NUMERIC_COLS,
    'categorical_cols': CAT_COLS,
    'category_values': category_values,
    'persona_policy': PERSONA_POLICY,
    'input_dim': int(x_train.shape[-1]),
    'd_model': D_MODEL,
    'n_heads': N_HEADS,
    'n_layers': N_LAYERS,
    'dim_feedforward': DIM_FEEDFORWARD,
    'dropout': DROPOUT,
    'seed': SEED,
    'max_rows': MAX_ROWS,
    'max_sequences': MAX_SEQUENCES,
    'history': history,
    'metrics': {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)},
}

torch.save({'state_dict': model.state_dict(), 'metadata': metadata}, model_path)
metadata_path.write_text(json.dumps(metadata, indent=2))

sample = df.loc[test_indices, ['timestamp', 'room_number', 'room_temp', 'setpoint', 'ideal_temp', 'outside_temp', 'hvac_mode', 'occupancy_prediction', 'temperature_persona_prediction']].copy().reset_index(drop=True)
sample['recommended_target'] = actual_setpoints
sample['model_prediction'] = pred_setpoints
sample.head(5000).to_csv(predictions_path, index=False)

report = '\n'.join([
    f'rows: {len(df):,}',
    f'sequences: {len(x):,}',
    f'train sequences: {len(x_train):,}',
    f'validation sequences: {len(x_val):,}',
    f'test sequences: {len(x_test):,}',
    f'sequence length: {SEQ_LEN}',
    f'prediction horizon minutes: {PREDICTION_HORIZON_MINUTES}',
    f'MAE: {mae:.3f}',
    f'RMSE: {rmse:.3f}',
    f'R2: {r2:.3f}',
])
report_path.write_text(report)
print(report)


rows: 5,212,900
sequences: 500,000
train sequences: 400,000
validation sequences: 50,000
test sequences: 50,000
sequence length: 24
prediction horizon minutes: 5
MAE: 0.217
RMSE: 0.310
R2: 0.962


In [14]:
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [model_path, metadata_path, report_path, predictions_path]:
        zf.write(path, arcname=path.name)

print('saved model:', model_path)
print('saved metadata:', metadata_path)
print('saved report:', report_path)
print('saved predictions:', predictions_path)
print('created zip:', zip_path)
print('You can download the zip from Google Drive:', zip_path)


saved model: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer.pt
saved metadata: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_metadata.json
saved report: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_report.txt
saved predictions: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_sample_predictions.csv
created zip: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_outputs.zip
You can download the zip from Google Drive: /content/drive/MyDrive/VisualizationApp/AIModelsAndAlgorithms/TempretureRecomendation/transformer/tempreture_recomendation_transformer_outputs.